# Quantization Playground — coding workshop (advanced)

Expected time: **35-45 minutes**, individual or pairs, CPU only.

This notebook stays deliberately small. The point is not to use a quantization library, but to see the arithmetic: scale, round, dequantize, measure the error, and then see why outliers and clipping matter.


## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import re

_solutions = {
    1: "import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nnp.set_printoptions(precision=4, suppress=True)\n\n\ndef absmax_quantize(x, bits=8):\n    x = np.asarray(x, dtype=np.float32)\n    qmax = 2 ** (bits - 1) - 1\n    alpha = float(np.max(np.abs(x)))\n    if alpha == 0:\n        return np.zeros_like(x, dtype=np.int32), 1.0\n    scale = qmax / alpha\n    q = np.round(x * scale)\n    q = np.clip(q, -qmax, qmax).astype(np.int32)\n    return q, scale\n\n\ndef absmax_dequantize(q, scale):\n    return np.asarray(q, dtype=np.float32) / scale\n\nvalues = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)\nq8, s8 = absmax_quantize(values, bits=8)\ndq8 = absmax_dequantize(q8, s8)\n\npd.DataFrame({\n    \"original\": values,\n    \"int8_code\": q8,\n    \"dequantized\": np.round(dq8, 4),\n    \"error\": np.round(values - dq8, 4),\n})",
    2: "rows = []\nfor bits in [8, 4, 3, 2]:\n    q, s = absmax_quantize(values, bits=bits)\n    dq = absmax_dequantize(q, s)\n    rows.append({\n        \"bits\": bits,\n        \"num_possible_signed_codes\": 2 ** bits,\n        \"num_codes_used_here\": len(set(q.tolist())),\n        \"scale\": s,\n        \"mean_abs_error\": np.mean(np.abs(values - dq)),\n        \"max_abs_error\": np.max(np.abs(values - dq)),\n    })\n\nbit_sweep = pd.DataFrame(rows)\nbit_sweep",
    3: "def asymmetric_quantize(x, bits=8):\n    x = np.asarray(x, dtype=np.float32)\n    qmin, qmax = 0, 2 ** bits - 1\n    xmin, xmax = float(np.min(x)), float(np.max(x))\n    if xmax == xmin:\n        return np.zeros_like(x, dtype=np.int32), 1.0, 0\n    scale = (xmax - xmin) / (qmax - qmin)\n    zero_point = int(np.round(qmin - xmin / scale))\n    q = np.round(x / scale + zero_point)\n    q = np.clip(q, qmin, qmax).astype(np.int32)\n    return q, scale, zero_point\n\n\ndef asymmetric_dequantize(q, scale, zero_point):\n    return (np.asarray(q, dtype=np.float32) - zero_point) * scale\n\nactivations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)\n\nq_sym, s_sym = absmax_quantize(activations, bits=4)\ndq_sym = absmax_dequantize(q_sym, s_sym)\n\nq_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)\ndq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)\n\npd.DataFrame({\n    \"activation\": activations,\n    \"sym_code\": q_sym,\n    \"sym_deq\": np.round(dq_sym, 3),\n    \"asym_code\": q_asym,\n    \"asym_deq\": np.round(dq_asym, 3),\n})",
    4: "def quantize_with_clipping(x, clip_range=5.0, bits=8):\n    clipped = np.clip(x, -clip_range, clip_range)\n    q, s = absmax_quantize(clipped, bits=bits)\n    dq = absmax_dequantize(q, s)\n    return clipped, q, dq, s\n\nsummary = []\nfor clip_range in [None, 10.0, 5.0, 2.5]:\n    if clip_range is None:\n        q, s = absmax_quantize(with_outlier, bits=8)\n        dq = absmax_dequantize(q, s)\n        label = \"no clipping\"\n    else:\n        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)\n        label = f\"clip [-{clip_range}, {clip_range}]\"\n    summary.append({\n        \"setting\": label,\n        \"ordinary_codes_used\": len(set(q[:-1].tolist())),\n        \"ordinary_MAE\": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),\n        \"all_values_MAE\": np.mean(np.abs(with_outlier - dq)),\n        \"outlier_dequantized\": dq[-1],\n    })\n\npd.DataFrame(summary)",
    5: "rng = np.random.default_rng(0)\nW = rng.normal(0, 0.02, size=(64, 256)).astype(np.float32)  # a small weight matrix\noutliers = rng.choice(W.size, size=20, replace=False)\nW.flat[outliers] *= 25  # a few large weights, as in real models\n\n\ndef groupwise_quantize(W, group_size, bits=4):\n    rows, cols = W.shape\n    assert cols % group_size == 0, \"the row length must be a multiple of the group size\"\n    groups = W.reshape(rows, cols // group_size, group_size)\n    qmax = 2 ** (bits - 1) - 1\n    alpha = np.max(np.abs(groups), axis=-1, keepdims=True)\n    scale = qmax / np.where(alpha == 0, 1.0, alpha)\n    q = np.clip(np.round(groups * scale), -qmax, qmax).astype(np.int32)\n    return q, scale\n\n\ndef groupwise_dequantize(q, scale):\n    return (q / scale).reshape(q.shape[0], -1).astype(np.float32)\n\nrows = []\nfor label, group_size in [(\"per tensor\", None), (\"per row\", 256), (\"groups of 64\", 64), (\"groups of 16\", 16)]:\n    if group_size is None:\n        q, s = absmax_quantize(W.ravel(), bits=4)\n        W_hat = absmax_dequantize(q, s).reshape(W.shape)\n    else:\n        q, s = groupwise_quantize(W, group_size, bits=4)\n        W_hat = groupwise_dequantize(q, s)\n    rows.append({\"scheme\": label, \"mean_abs_error\": float(np.mean(np.abs(W - W_hat)))})\ngroup_errors = pd.DataFrame(rows)\ngroup_errors",
    6: "def bits_per_weight(bits, group_size, scale_bits=16):\n    return bits + scale_bits / group_size\n\n\ndef best_under_budget(table, max_bits_per_weight):\n    allowed = table[table[\"bits_per_weight\"] <= max_bits_per_weight]\n    return allowed.loc[allowed[\"mean_abs_error\"].idxmin()]\n\nrows = []\nfor bits in [8, 4, 3, 2]:\n    for group_size in [256, 128, 64, 32, 16]:\n        q, s = groupwise_quantize(W, group_size, bits=bits)\n        rows.append({\n            \"bits\": bits,\n            \"group_size\": group_size,\n            \"bits_per_weight\": bits_per_weight(bits, group_size),\n            \"mean_abs_error\": float(np.mean(np.abs(W - groupwise_dequantize(q, s)))),\n        })\nbudget = pd.DataFrame(rows)\n\nfor limit in [4.5, 3.5]:\n    best = best_under_budget(budget, limit)\n    print(f\"Best under {limit} bits per weight: {int(best['bits'])}-bit, groups of \"\n          f\"{int(best['group_size'])} ({best['bits_per_weight']:.3f} bits/weight, \"\n          f\"error {best['mean_abs_error']:.5f})\")\nprint(f\"\\n1.54B parameters at 4 bits, groups of 64: {1.54e9 * bits_per_weight(4, 64) / 8 / 1024**3:.2f} GB\")\nbudget.sort_values(\"bits_per_weight\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution."""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if not m or int(m.group(1)) not in _fill_in:
        return lines
    n = int(m.group(1))
    return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Part A - absmax quantization from scratch

Implement the symmetric absmax method from the lesson. The range is centered on zero, and the largest absolute value defines the scale.


### ✏️ Exercise 1 — Absmax quantization from scratch

Complete `absmax_quantize` and `absmax_dequantize`.

- `qmax` is the largest positive code of a signed `bits`-bit integer (127 for 8 bits).
- `alpha` is the largest absolute value in `x`.
- The scale maps `alpha` to `qmax`; each value is multiplied by it and rounded.
- Dequantizing divides the codes by the scale.

<details><summary><b>Hints</b> (click to open)</summary>

- A signed `bits`-bit integer has codes from −(2^(bits−1) − 1) to 2^(bits−1) − 1.
- scale = qmax / alpha, so the largest value lands exactly on qmax.
- Quantize: multiply by the scale. Dequantize: divide by it.

</details>

In [ ]:
# Exercise 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def absmax_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmax = ___  # TODO: largest positive code of a signed `bits`-bit integer
    alpha = ___  # TODO: largest absolute value in x
    if alpha == 0:
        return np.zeros_like(x, dtype=np.int32), 1.0
    scale = ___  # TODO
    q = ___  # TODO: scaled and rounded (clipping comes next)
    q = np.clip(q, -qmax, qmax).astype(np.int32)
    return q, scale


def absmax_dequantize(q, scale):
    # TODO: map the integer codes back to floats
    return ___

values = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)
q8, s8 = absmax_quantize(values, bits=8)
dq8 = absmax_dequantize(q8, s8)

pd.DataFrame({
    "original": values,
    "int8_code": q8,
    "dequantized": np.round(dq8, 4),
    "error": np.round(values - dq8, 4),
})


In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(s8 - 127 / 10.8) < 1e-4, f"scale should be 127 / 10.8, got {s8}"
assert q8.tolist() == [64, 36, -89, 0, -22, -54, 127, 36, -23], q8.tolist()
assert np.allclose(dq8, q8 / s8)
_q4, _ = absmax_quantize(values, bits=4)
assert _q4.max() == 7, "4-bit signed codes go up to 7"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


def absmax_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmax = 2 ** (bits - 1) - 1
    alpha = float(np.max(np.abs(x)))
    if alpha == 0:
        return np.zeros_like(x, dtype=np.int32), 1.0
    scale = qmax / alpha
    q = np.round(x * scale)
    q = np.clip(q, -qmax, qmax).astype(np.int32)
    return q, scale


def absmax_dequantize(q, scale):
    return np.asarray(q, dtype=np.float32) / scale

values = np.array([5.47, 3.08, -7.59, 0.0, -1.85, -4.57, 10.8, 3.02, -1.92], dtype=np.float32)
q8, s8 = absmax_quantize(values, bits=8)
dq8 = absmax_dequantize(q8, s8)

pd.DataFrame({
    "original": values,
    "int8_code": q8,
    "dequantized": np.round(dq8, 4),
    "error": np.round(values - dq8, 4),
})
```

</details>

In [ ]:
print("scale:", round(s8, 4))
print("original:    ", values)
print("quantized:   ", q8)
print("dequantized: ", np.round(dq8, 4))
print("mean absolute error:", np.mean(np.abs(values - dq8)))

for v, q, d in zip(values, q8, dq8):
    if q == 36:
        print(f"{v} -> {q} -> {d:.4f}")


### Reflection

What makes collisions more likely, where two original values become the same integer code? Write one or two sentences before moving on.


## Part B - sweep over bit widths

The same vector is quantized with 8, 4, 3, and 2 bits. This shows why very low precision usually needs more careful methods than plain rounding.


### ✏️ Exercise 2 — Sweep the bit widths

For each bit width, fill in three columns: how many distinct codes this vector
actually uses, and the mean and maximum absolute reconstruction error.

<details><summary><b>Hints</b> (click to open)</summary>

- `set(...)` keeps each distinct value once; `len` counts them. Turn `q` into a list first.
- The error of each value is `values - dq`. Take its absolute value first.

</details>

In [ ]:
# Exercise 2
rows = []
for bits in [8, 4, 3, 2]:
    q, s = absmax_quantize(values, bits=bits)
    dq = absmax_dequantize(q, s)
    rows.append({
        "bits": bits,
        "num_possible_signed_codes": 2 ** bits,
        "num_codes_used_here": ___,  # TODO: number of distinct codes in q
        "scale": s,
        "mean_abs_error": ___,  # TODO: mean of |original - dequantized|
        "max_abs_error": ___,  # TODO: largest |original - dequantized|
    })

bit_sweep = pd.DataFrame(rows)
bit_sweep


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert list(bit_sweep["bits"]) == [8, 4, 3, 2]
assert bit_sweep.loc[0, "num_codes_used_here"] == 8, "at 8 bits, 3.08 and 3.02 share code 36"
assert bit_sweep.loc[0, "mean_abs_error"] < bit_sweep.loc[3, "mean_abs_error"]
assert (bit_sweep["max_abs_error"] >= bit_sweep["mean_abs_error"]).all()
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
rows = []
for bits in [8, 4, 3, 2]:
    q, s = absmax_quantize(values, bits=bits)
    dq = absmax_dequantize(q, s)
    rows.append({
        "bits": bits,
        "num_possible_signed_codes": 2 ** bits,
        "num_codes_used_here": len(set(q.tolist())),
        "scale": s,
        "mean_abs_error": np.mean(np.abs(values - dq)),
        "max_abs_error": np.max(np.abs(values - dq)),
    })

bit_sweep = pd.DataFrame(rows)
bit_sweep
```

</details>

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(bit_sweep["bits"], bit_sweep["mean_abs_error"], marker="o")
plt.gca().invert_xaxis()
plt.xlabel("bits")
plt.ylabel("mean absolute reconstruction error")
plt.title("Lower precision usually increases reconstruction error")
plt.show()


## Part C - asymmetric quantization for lopsided data

A quick reminder of what "activations" are: the intermediate output values produced as
data flows through a network's layers during a forward pass, not the weights. Weights are
fixed after training; activations are computed fresh from whatever input the model is
currently given, so their range can shift from one input to the next.

Here the values are all non-negative, like activations after a ReLU (ReLU zeroes out
anything negative, so everything downstream of it is $\geq 0$). Symmetric quantization
wastes half the code range on negative values that never occur.


### ✏️ Exercise 3 — Asymmetric quantization

Complete `asymmetric_quantize` and `asymmetric_dequantize`. The codes run from 0 to
2^bits − 1. The range [xmin, xmax] is mapped onto them, and the **zero point** is the code
that represents 0.0.

<details><summary><b>Hints</b> (click to open)</summary>

- The scale is the width of the range divided by the number of steps.
- Quantize: divide by the scale, then shift by the zero point.
- Dequantize: undo the shift, then multiply by the scale.

</details>

In [ ]:
# Exercise 3
def asymmetric_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmin, qmax = 0, 2 ** bits - 1
    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmax == xmin:
        return np.zeros_like(x, dtype=np.int32), 1.0, 0
    scale = ___  # TODO: size of one step, in the units of x
    zero_point = ___  # TODO: the code that represents 0.0, as an int
    q = ___  # TODO: codes before clipping
    q = np.clip(q, qmin, qmax).astype(np.int32)
    return q, scale, zero_point


def asymmetric_dequantize(q, scale, zero_point):
    # TODO: map the codes back to floats
    return ___

activations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)

q_sym, s_sym = absmax_quantize(activations, bits=4)
dq_sym = absmax_dequantize(q_sym, s_sym)

q_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)
dq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)

pd.DataFrame({
    "activation": activations,
    "sym_code": q_sym,
    "sym_deq": np.round(dq_sym, 3),
    "asym_code": q_asym,
    "asym_deq": np.round(dq_asym, 3),
})


In [ ]:
# Check for Exercise 3. Run this cell as is.
assert q_asym.min() == 0 and q_asym.max() == 15, q_asym
assert z_asym == 0, "these activations start at 0.0, so code 0 represents 0.0"
_q, _s, _z = asymmetric_quantize(np.array([-1.0, 3.0]), bits=8)
assert (_q.tolist(), _z) == ([0, 255], 64), (_q.tolist(), _z)
assert np.mean(np.abs(activations - dq_asym)) < np.mean(np.abs(activations - dq_sym))
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def asymmetric_quantize(x, bits=8):
    x = np.asarray(x, dtype=np.float32)
    qmin, qmax = 0, 2 ** bits - 1
    xmin, xmax = float(np.min(x)), float(np.max(x))
    if xmax == xmin:
        return np.zeros_like(x, dtype=np.int32), 1.0, 0
    scale = (xmax - xmin) / (qmax - qmin)
    zero_point = int(np.round(qmin - xmin / scale))
    q = np.round(x / scale + zero_point)
    q = np.clip(q, qmin, qmax).astype(np.int32)
    return q, scale, zero_point


def asymmetric_dequantize(q, scale, zero_point):
    return (np.asarray(q, dtype=np.float32) - zero_point) * scale

activations = np.array([0.0, 0.1, 0.4, 0.9, 1.0, 1.4, 2.0, 2.2, 2.8], dtype=np.float32)

q_sym, s_sym = absmax_quantize(activations, bits=4)
dq_sym = absmax_dequantize(q_sym, s_sym)

q_asym, s_asym, z_asym = asymmetric_quantize(activations, bits=4)
dq_asym = asymmetric_dequantize(q_asym, s_asym, z_asym)

pd.DataFrame({
    "activation": activations,
    "sym_code": q_sym,
    "sym_deq": np.round(dq_sym, 3),
    "asym_code": q_asym,
    "asym_deq": np.round(dq_asym, 3),
})
```

</details>

In [ ]:
print("Symmetric mean abs error:", np.mean(np.abs(activations - dq_sym)))
print("Asymmetric mean abs error:", np.mean(np.abs(activations - dq_asym)))
print("Asymmetric scale:", s_asym, "zero point:", z_asym)


## Part D - outliers and clipping

Absmax quantization can waste most of the integer range on one extreme value. Clipping deliberately damages the outlier so the ordinary values are represented better.


In [ ]:
rng = np.random.default_rng(7)
ordinary = np.round(rng.uniform(-2, 2, size=24), 2)
with_outlier = np.concatenate([ordinary, [256.0]])

q_out, s_out = absmax_quantize(with_outlier, bits=8)
dq_out = absmax_dequantize(q_out, s_out)

ordinary_codes = q_out[:-1]
print("Distinct codes used by ordinary values, no clipping:", len(set(ordinary_codes.tolist())))
print("First 10 ordinary values/codes:")
print(list(zip(with_outlier[:10], q_out[:10])))


### ✏️ Exercise 4 — Clip before quantizing

Complete `quantize_with_clipping`: limit every value to the range
[−clip_range, clip_range] before absmax quantization.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.clip(x, low, high)` replaces values below `low` with `low` and above `high` with `high`.

</details>

In [ ]:
# Exercise 4
def quantize_with_clipping(x, clip_range=5.0, bits=8):
    clipped = ___  # TODO: x limited to [-clip_range, clip_range]
    q, s = absmax_quantize(clipped, bits=bits)
    dq = absmax_dequantize(q, s)
    return clipped, q, dq, s

summary = []
for clip_range in [None, 10.0, 5.0, 2.5]:
    if clip_range is None:
        q, s = absmax_quantize(with_outlier, bits=8)
        dq = absmax_dequantize(q, s)
        label = "no clipping"
    else:
        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)
        label = f"clip [-{clip_range}, {clip_range}]"
    summary.append({
        "setting": label,
        "ordinary_codes_used": len(set(q[:-1].tolist())),
        "ordinary_MAE": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),
        "all_values_MAE": np.mean(np.abs(with_outlier - dq)),
        "outlier_dequantized": dq[-1],
    })

pd.DataFrame(summary)


In [ ]:
# Check for Exercise 4. Run this cell as is.
_c, _q, _dq, _s = quantize_with_clipping(np.array([-10.0, 0.0, 1.0, 10.0]), clip_range=2.0)
assert _c.tolist() == [-2, 0, 1, 2], _c.tolist()
assert _q.tolist() == [-127, 0, 64, 127], _q.tolist()
_df = pd.DataFrame(summary)
assert _df["ordinary_codes_used"].iloc[-1] > _df["ordinary_codes_used"].iloc[0]
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def quantize_with_clipping(x, clip_range=5.0, bits=8):
    clipped = np.clip(x, -clip_range, clip_range)
    q, s = absmax_quantize(clipped, bits=bits)
    dq = absmax_dequantize(q, s)
    return clipped, q, dq, s

summary = []
for clip_range in [None, 10.0, 5.0, 2.5]:
    if clip_range is None:
        q, s = absmax_quantize(with_outlier, bits=8)
        dq = absmax_dequantize(q, s)
        label = "no clipping"
    else:
        _, q, dq, s = quantize_with_clipping(with_outlier, clip_range=clip_range, bits=8)
        label = f"clip [-{clip_range}, {clip_range}]"
    summary.append({
        "setting": label,
        "ordinary_codes_used": len(set(q[:-1].tolist())),
        "ordinary_MAE": np.mean(np.abs(with_outlier[:-1] - dq[:-1])),
        "all_values_MAE": np.mean(np.abs(with_outlier - dq)),
        "outlier_dequantized": dq[-1],
    })

pd.DataFrame(summary)
```

</details>

### ✏️ Exercise 5 — Group-wise quantization (advanced)

Part D quantized a whole vector with **one** scale, so a single outlier sets the
scale for everything. Real 4-bit formats split each row of a weight matrix into small
**groups** of consecutive weights, and give each group its own absmax scale. An outlier
then only hurts the few weights in its own group. For example, the NF4 format of QLoRA
(Dettmers et al. 2023) uses groups of 64 weights.

For a group g with values x, the same absmax formulas as Part A apply, group by group:

scale_g = qmax / max|x|,  q = round(x · scale_g),  x̂ = q / scale_g

Complete `groupwise_quantize` and `groupwise_dequantize` without a Python loop: reshape
the matrix to (rows, groups per row, group size) and let NumPy work on the last axis. The
cell compares one scale for the whole matrix, one per row, and groups of 64 and 16 on a
matrix with a few outliers.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.max(np.abs(groups), axis=-1, keepdims=True)` takes the max over each group and keeps a size-1 last axis.
- Because `scale` has shape (rows, groups, 1), `groups * scale` multiplies each group by its own scale.
- `q / scale` has the grouped shape; `.reshape(q.shape[0], -1)` puts each row back together.

</details>

In [ ]:
# Exercise 5
rng = np.random.default_rng(0)
W = rng.normal(0, 0.02, size=(64, 256)).astype(np.float32)  # a small weight matrix
outliers = rng.choice(W.size, size=20, replace=False)
W.flat[outliers] *= 25  # a few large weights, as in real models


def groupwise_quantize(W, group_size, bits=4):
    rows, cols = W.shape
    assert cols % group_size == 0, "the row length must be a multiple of the group size"
    groups = W.reshape(rows, cols // group_size, group_size)
    qmax = 2 ** (bits - 1) - 1
    alpha = ___  # TODO: max |value| of each group, shape (rows, groups per row, 1)
    scale = qmax / np.where(alpha == 0, 1.0, alpha)
    q = ___  # TODO: round, clip to [-qmax, qmax], convert to integers
    return q, scale


def groupwise_dequantize(q, scale):
    # TODO: undo the scaling and reshape back to (rows, cols)
    ___

rows = []
for label, group_size in [("per tensor", None), ("per row", 256), ("groups of 64", 64), ("groups of 16", 16)]:
    if group_size is None:
        q, s = absmax_quantize(W.ravel(), bits=4)
        W_hat = absmax_dequantize(q, s).reshape(W.shape)
    else:
        q, s = groupwise_quantize(W, group_size, bits=4)
        W_hat = groupwise_dequantize(q, s)
    rows.append({"scheme": label, "mean_abs_error": float(np.mean(np.abs(W - W_hat)))})
group_errors = pd.DataFrame(rows)
group_errors

In [ ]:
# Check for Exercise 5. Run this cell as is.
_q, _s = groupwise_quantize(W, 256, bits=4)
for _r in [0, 7, 63]:
    _qr, _sr = absmax_quantize(W[_r], bits=4)
    assert np.array_equal(_q[_r].ravel(), _qr), "one group per row should match absmax_quantize on that row"
assert groupwise_dequantize(_q, _s).shape == W.shape
_err = list(group_errors["mean_abs_error"])
assert _err[0] > _err[1] > _err[2] > _err[3], "smaller groups should give smaller errors here"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
rng = np.random.default_rng(0)
W = rng.normal(0, 0.02, size=(64, 256)).astype(np.float32)  # a small weight matrix
outliers = rng.choice(W.size, size=20, replace=False)
W.flat[outliers] *= 25  # a few large weights, as in real models


def groupwise_quantize(W, group_size, bits=4):
    rows, cols = W.shape
    assert cols % group_size == 0, "the row length must be a multiple of the group size"
    groups = W.reshape(rows, cols // group_size, group_size)
    qmax = 2 ** (bits - 1) - 1
    alpha = np.max(np.abs(groups), axis=-1, keepdims=True)
    scale = qmax / np.where(alpha == 0, 1.0, alpha)
    q = np.clip(np.round(groups * scale), -qmax, qmax).astype(np.int32)
    return q, scale


def groupwise_dequantize(q, scale):
    return (q / scale).reshape(q.shape[0], -1).astype(np.float32)

rows = []
for label, group_size in [("per tensor", None), ("per row", 256), ("groups of 64", 64), ("groups of 16", 16)]:
    if group_size is None:
        q, s = absmax_quantize(W.ravel(), bits=4)
        W_hat = absmax_dequantize(q, s).reshape(W.shape)
    else:
        q, s = groupwise_quantize(W, group_size, bits=4)
        W_hat = groupwise_dequantize(q, s)
    rows.append({"scheme": label, "mean_abs_error": float(np.mean(np.abs(W - W_hat)))})
group_errors = pd.DataFrame(rows)
group_errors
```

</details>

### ✏️ Exercise 6 — Rounding error against bits stored (advanced)

Each group also stores its scale, usually as a 16-bit float. So a group-wise format
costs more than `bits` per weight:

bits per weight = bits + 16 / group_size

Smaller groups give smaller errors, but store more scales. Complete `bits_per_weight` and
`best_under_budget`. The cell tries every combination of 8, 4, 3 and 2 bits with group
sizes from 256 to 16, then picks the combination with the smallest error that fits
within 4.5 and within 3.5 bits per weight.

**Reflection:** under 3.5 bits per weight, is it better to use 3 bits with small groups,
or 2 bits with even smaller groups? What does the table say?

<details><summary><b>Hints</b> (click to open)</summary>

- A scale of 16 bits shared by `group_size` weights costs `16 / group_size` bits per weight.
- `table[table["bits_per_weight"] <= limit]` keeps the allowed rows.
- `allowed.loc[allowed["mean_abs_error"].idxmin()]` is the row with the smallest error.

</details>

In [ ]:
# Exercise 6
def bits_per_weight(bits, group_size, scale_bits=16):
    # TODO: the code bits plus each weight's share of its group's scale
    ___


def best_under_budget(table, max_bits_per_weight):
    # TODO: among the rows with bits_per_weight <= max_bits_per_weight,
    # return the row (a pandas Series) with the smallest mean_abs_error
    ___

rows = []
for bits in [8, 4, 3, 2]:
    for group_size in [256, 128, 64, 32, 16]:
        q, s = groupwise_quantize(W, group_size, bits=bits)
        rows.append({
            "bits": bits,
            "group_size": group_size,
            "bits_per_weight": bits_per_weight(bits, group_size),
            "mean_abs_error": float(np.mean(np.abs(W - groupwise_dequantize(q, s)))),
        })
budget = pd.DataFrame(rows)

for limit in [4.5, 3.5]:
    best = best_under_budget(budget, limit)
    print(f"Best under {limit} bits per weight: {int(best['bits'])}-bit, groups of "
          f"{int(best['group_size'])} ({best['bits_per_weight']:.3f} bits/weight, "
          f"error {best['mean_abs_error']:.5f})")
print(f"\n1.54B parameters at 4 bits, groups of 64: {1.54e9 * bits_per_weight(4, 64) / 8 / 1024**3:.2f} GB")
budget.sort_values("bits_per_weight")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert bits_per_weight(4, 64) == 4.25 and bits_per_weight(4, 16) == 5.0
_b = best_under_budget(budget, 4.5)
assert _b["bits_per_weight"] <= 4.5
assert _b["mean_abs_error"] == budget[budget["bits_per_weight"] <= 4.5]["mean_abs_error"].min()
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
def bits_per_weight(bits, group_size, scale_bits=16):
    return bits + scale_bits / group_size


def best_under_budget(table, max_bits_per_weight):
    allowed = table[table["bits_per_weight"] <= max_bits_per_weight]
    return allowed.loc[allowed["mean_abs_error"].idxmin()]

rows = []
for bits in [8, 4, 3, 2]:
    for group_size in [256, 128, 64, 32, 16]:
        q, s = groupwise_quantize(W, group_size, bits=bits)
        rows.append({
            "bits": bits,
            "group_size": group_size,
            "bits_per_weight": bits_per_weight(bits, group_size),
            "mean_abs_error": float(np.mean(np.abs(W - groupwise_dequantize(q, s)))),
        })
budget = pd.DataFrame(rows)

for limit in [4.5, 3.5]:
    best = best_under_budget(budget, limit)
    print(f"Best under {limit} bits per weight: {int(best['bits'])}-bit, groups of "
          f"{int(best['group_size'])} ({best['bits_per_weight']:.3f} bits/weight, "
          f"error {best['mean_abs_error']:.5f})")
print(f"\n1.54B parameters at 4 bits, groups of 64: {1.54e9 * bits_per_weight(4, 64) / 8 / 1024**3:.2f} GB")
budget.sort_values("bits_per_weight")
```

</details>

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(np.arange(len(with_outlier)), with_outlier, label="original")
_, _, dq_clip5, _ = quantize_with_clipping(with_outlier, clip_range=5.0, bits=8)
plt.scatter(np.arange(len(dq_clip5)), dq_clip5, marker="x", label="after clip+quantize+dequantize")
plt.xlabel("index")
plt.ylabel("value")
plt.title("Clipping preserves ordinary values better but damages the outlier")
plt.legend()
plt.show()


### Reflection

Was clipping worth it here? Answer twice: once if the ordinary values matter most, and once if the outlier matters just as much as every other value.


## Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow. At minimum, comment on:

1. how the scale is computed;
2. why rounding creates quantization error;
3. why clipping helps ordinary values in the outlier example.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.
